In [1]:
import pandas as pd
import joblib

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")['Churn']
y_test = pd.read_parquet("../data/processed/y_test.parquet")['Churn']
preprocessor = joblib.load("../data/processed/preprocessor.joblib")

## Model Comparison - LogisticRegression vs RandomForest Classifier

Baseline comparison of both models ('class_weight='balanced', from previous decision) using 5-folds `StratifiedKFold` cross-validation on the training set. F1 and recall (minority class) are tracked

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42))
])

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_scores = cross_validate(logreg_pipeline, X_train, y_train, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], return_train_score=True)

rf_scores = cross_validate(rf_pipeline, X_train, y_train, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], return_train_score=True)

In [4]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [logreg_scores['test_accuracy'].mean(), rf_scores['test_accuracy'].mean()],
    'Precision': [logreg_scores['test_precision'].mean(), rf_scores['test_precision'].mean()],
    'Recall': [logreg_scores['test_recall'].mean(), rf_scores['test_recall'].mean()],
    'F1-Score': [logreg_scores['test_f1'].mean(), rf_scores['test_f1'].mean()]
})

print(comparison_df)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.749914   0.518611  0.803344  0.630239
1        Random Forest  0.790916   0.646413  0.469565  0.543921


**Findings:** LogisticRegression catches 80% of churners; RandomForest catches 47%. That's the number that decides this, a missed churner causes no chance of intervention, so recall matters more so than accuracy or precision RandomForest wins on (79.1% vs 75.0% accuracy, 0.646 vs 0.519 precision). F1 actually favors RandomForest (0.544 vs 0.630 for LogReg), but F1 is the wrong lens here as it rewards a precision/recall balance this problem doesn't need.

The gap probably comes down to how `class_weight='balanced'` hits each model differently. LogisticRegression reweights the loss directly, so it's a strong, blunt push toward flagging minority-class rows. RandomForest reweights impurity at each split instead, and that effect thins out once you're averaging votes across 100 trees. Neither model is tuned yet, so this isn't the two algorithms at their best but at default setting up

**Decision:** LogisticRegression is the model going forward. RandomForest stays on the
bench. If tuning LogReg stalls, or if false positives start costing more than assumed,
it's the first thing to revisit.

## Hyperparameter Tuning

Tuned LogisticRegression's `C`, `penalty`, and `solver` with `GridSearchCV`, same 5-fold `StratifiedKFold` as Step 6, scoring on recall per the Step 4 decision. `lbfgs` only supports `l2`, so the grid split into two subspaces (`lbfgs`+`l2`, and `liblinear`/`saga`+`l1`/`l2`). 25 valid combos total — small enough to search exhaustively instead of sampling.

In [5]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_distributions = [
    {
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "classifier__solver": ["liblinear", "saga"],
        "classifier__penalty": ["l1", "l2"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
]

In [6]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    logreg_pipeline,
    param_grid=param_distributions,
    cv=cv,
    scoring="recall",
    return_train_score=True,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'classifier__C': 0.1, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
0.8127090301003344


In [7]:
tuned_scores = cross_validate(
    grid_search.best_estimator_, 
    X_train, 
    y_train, 
    cv=cv, 
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], 
    return_train_score=True
)

print({k: v.mean() for k, v in tuned_scores.items() if 'test_' in k})

grid_search.best_estimator_.named_steps['classifier'].coef_

{'test_accuracy': np.float64(0.7518673729434626), 'test_precision': np.float64(0.5210822748457613), 'test_recall': np.float64(0.8127090301003344), 'test_f1': np.float64(0.6349150415824765), 'test_roc_auc': np.float64(0.8508100378249619)}


array([[-5.01177170e-01,  3.84379201e-01, -2.75062345e-02,
         3.74259422e-01,  0.00000000e+00,  0.00000000e+00,
         1.35082963e-01,  0.00000000e+00, -1.49842181e-01,
        -5.57538132e-01,  0.00000000e+00,  2.16377444e-01,
         5.93812106e-01, -9.92138497e-04, -5.23648403e-01,
        -3.67870985e-01, -2.74523958e-07, -1.19271360e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        -2.92071129e-01, -2.98729430e-03,  1.41513700e-01,
        -5.63393523e-04,  1.56280623e-01, -6.42953334e-01,
        -1.42823280e+00,  3.23823329e-01,  0.00000000e+00,
         3.80498694e-01,  0.00000000e+00,  0.00000000e+00,
        -2.08610200e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00]])

Best params: `C=0.1`, `penalty='l1'`, `solver='liblinear'`.

| Metric | Step 6 baseline | Tuned |
|---|---|---|
| Accuracy | 0.750 | 0.752 |
| Precision | 0.519 | 0.521 |
| Recall | 0.803 | 0.813 |
| F1 | 0.630 | 0.635 |

All four metrics improved, not just recall. Train and test scores are close on every metric (recall 0.813 both), so the regularization isn't over- or under-fitting.

In [8]:
feature_names = grid_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()
coefs = grid_search.best_estimator_.named_steps['classifier'].coef_[0]
pd.Series(coefs, index=feature_names).sort_values()

cat__Contract_Two year                       -1.428233e+00
cat__Contract_One year                       -6.429533e-01
cat__PhoneService_Yes                        -5.575381e-01
cat__OnlineSecurity_No internet service      -5.236484e-01
num__tenure                                  -5.011772e-01
cat__OnlineSecurity_Yes                      -3.678710e-01
cat__TechSupport_Yes                         -2.920711e-01
cat__tenure_bucket_24-35                     -2.086102e-01
cat__Dependents_Yes                          -1.498422e-01
cat__OnlineBackup_Yes                        -1.192714e-01
num__TotalCharges                            -2.750623e-02
cat__StreamingTV_No internet service         -2.987294e-03
cat__InternetService_No                      -9.921385e-04
cat__StreamingMovies_No internet service     -5.633935e-04
cat__OnlineBackup_No internet service        -2.745240e-07
cat__PaymentMethod_Mailed check               0.000000e+00
cat__PaymentMethod_Credit card (automatic)    0.000000e+

`l1` zeroed 11 of 37 coefficients, including two of six `..._No internet service` dummies (`DeviceProtection`, `StreamingMovies`). Those six columns duplicate each other, all flip to 1 together when a customer has no internet service, so `l1` collapsed real redundancy, not just weak signal.

Largest coefficients: `Contract_Two year` (-1.43), `Contract_One year` (-0.64), `tenure` (-0.50), `InternetService_Fiber optic` (+0.59). Contract length dominates the model.

**Decision:** tuned LogisticRegression (`C=0.1`, `l1`, `liblinear`) replaces the Step 6 baseline. RandomForest stays the untuned fallback.

## Boosting Comparison (XGBoost)

Benchmarks are gradient-boosted tree model against the LogisticRegressiona and RandomForest using the same `StratifiedKFold` CV and scoring set

**Class Weighting:** LogisticRegression and RandomForest use `class_weight='balanced'`, but XGBoost doesn't use `class_weight`. The equivalent is instead `scale_pos_weight = n_negative / n_positive`, made using `y_train`

The tuned LogisticRegression models will remain primary model unless XGBoost model, beats it on **both recall and ROC-AUC** in the cross-validation. RandomForest stays the fallback option if needed.



In [9]:
from xgboost import XGBClassifier

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        scale_pos_weight=scale_pos_weight, 
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    ))
])


In [10]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

xgb_scores = cross_validate(xgb_pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)

print({k: v.mean() for k, v in xgb_scores.items() if 'test_' in k})

{'test_accuracy': np.float64(0.7703245542560216), 'test_precision': np.float64(0.5562861499481218), 'test_recall': np.float64(0.6642140468227424), 'test_f1': np.float64(0.6054372811933525), 'test_roc_auc': np.float64(0.8221208370861044)}


In [11]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression (tuned)', 'Random Forest', 'XGBoost'],
    'Accuracy':  [tuned_scores['test_accuracy'].mean(),  rf_scores['test_accuracy'].mean(),  xgb_scores['test_accuracy'].mean()],
    'Precision': [tuned_scores['test_precision'].mean(), rf_scores['test_precision'].mean(), xgb_scores['test_precision'].mean()],
    'Recall':    [tuned_scores['test_recall'].mean(),    rf_scores['test_recall'].mean(),    xgb_scores['test_recall'].mean()],
    'F1-Score':  [tuned_scores['test_f1'].mean(),        rf_scores['test_f1'].mean(),        xgb_scores['test_f1'].mean()],
    'ROC-AUC':   [tuned_scores['test_roc_auc'].mean(),   rf_scores['test_roc_auc'].mean(),   xgb_scores['test_roc_auc'].mean()],
})
print(comparison_df)

                         Model  Accuracy  Precision    Recall  F1-Score  \
0  Logistic Regression (tuned)  0.751867   0.521082  0.812709  0.634915   
1                Random Forest  0.790916   0.646413  0.469565  0.543921   
2                      XGBoost  0.770325   0.556286  0.664214  0.605437   

    ROC-AUC  
0  0.850810  
1  0.824984  
2  0.822121  


In [13]:
from sklearn.model_selection import RandomizedSearchCV

xgb_param_distributions = {
    "classifier__n_estimators":     [100, 200, 400, 600],
    "classifier__learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "classifier__max_depth":        [3, 4, 5, 6],
    "classifier__subsample":        [0.7, 0.85, 1.0],
    "classifier__colsample_bytree": [0.7, 0.85, 1.0],
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=20,
    cv=cv,
    scoring='recall',          # consistent with Step 4
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print(xgb_search.best_params_)
print(xgb_search.best_score_)

# full metric set on the tuned booster, same as the other models
xgb_tuned_scores = cross_validate(
    xgb_search.best_estimator_, X_train, y_train, cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True
)
print({k: v.mean() for k, v in xgb_tuned_scores.items() if k.startswith('test_')})

{'classifier__subsample': 0.85, 'classifier__n_estimators': 200, 'classifier__max_depth': 4, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 1.0}
0.8080267558528428
{'test_accuracy': np.float64(0.7429920520219826), 'test_precision': np.float64(0.509963305074404), 'test_recall': np.float64(0.8107023411371237), 'test_f1': np.float64(0.6260539758669037), 'test_roc_auc': np.float64(0.8465299555180762)}


**Findings:** Baseline XGBoost (`scale_pos_weight`, otherwise default) landed at recall
0.664 / ROC-AUC 0.822, well behind the tuned LogisticRegression. A 20-iteration
`RandomizedSearchCV` on recall closed the recall gap (0.664 -> 0.811) but produced no
metric that beats tuned LogReg: recall 0.811 vs 0.813, ROC-AUC 0.847 vs 0.851,
precision 0.510 vs 0.521, F1 0.626 vs 0.635, accuracy 0.743 vs 0.752.

The tuned booster's best config, `learning_rate=0.01`, `max_depth=4`,
`n_estimators=200`, is a shallow, strongly regularised model, i.e. the search drove
XGBoost toward a near-linear decision boundary to match the linear model rather than
finding extra non-linear signal.

**Decision:** tuned LogisticRegression (`C=0.1`, `l1`, `liblinear`) remains the primary
model. It is not beaten on any metric, and it is the model whose coefficients are
directly interpretable for the business narrative. RandomForest and XGBoost stay
as alternatives.